# v3.5 link C — the head-crop arms, end to end on this A100

Six arms over the **51 pairs either failure record marks hard** (`v35_linkC.csv`), seeds **46/47/48**. Everything runs here: the crops, the parsing, the klein calls, the upscale. Nothing is done on a laptop between sessions.

| arm | call 1 | head off | ankle cut | computed here |
|---|---|---|---|---|
| `VEi` | the lock's `Q3` — mannequin head **and** re-pose | no | no | no — reused |
| `BC` | v3.1's incumbent: bald pass → V2 crop | the V2 cropper | no | no — reused |
| `VEic` | the lock's `Q3` | a crop | no | yes* |
| `M1qbc` | `Q3` minus the mannequin sentence, **plus bald** | a crop | no | yes |
| `VEica` | as `VEic` | a crop | **yes** | yes* |
| `M1qbca` | as `M1qbc` | a crop | **yes** | yes |

\* reused too, if a previous `v35_linkC_*.zip` is on Drive.

**Why the re-pose arm is bald.** `BC` bald-passes the raw photograph before it crops, because hair on the shoulders and chest cannot be told from garment by any matte; `VEi` gets that free, since the mannequin sentence takes the head and its hair together. A re-pose arm that keeps the wearer's own head does neither — cropping it leaves whatever hair spilled onto the garment. So **`M1q` without the bald clause is not a shippable arm and is not run**: the bald version covers that case and is the only functional form. One klein call does both jobs, measured on the five longest-haired garments of the fold (`v3/report/v35_bald.html`).

An arm name is read, not looked up: base (`VEi` | `M1qb`) + `c` head crop + `a` ankle cut.

**Order of operations, which is not the obvious one:**

    call 1 ──▶ white-margin re-crop ──▶ HEAD CROP ──▶ [ankle cut] ──▶ SR to ~1 MP ──▶ call 2

The crops go **before** the SR pass. Cropping afterwards would take the reference back below 1 MP and break the rule v3.4 link H bought — what conditioning contributes is bounded by its **token footprint** in call 2. A cropped reference is a smaller image and has to be re-floated, or the arm tests two changes at once.

Neither crop is new code: the head crop is `ironman_bc_crop.crop_bc`, the call that makes the `BC` references, and the ankle cut is `run_ironman.ankle_cut`, v3.3's, verbatim (v3.4 removed it, so it is reopened here as its own variable and each `a` arm is paired with its cut-less twin).

**Cell 6 is a GPU audit** — it asserts every stage is on the GPU, including the two ONNX models that V2 hard-coded to CPU. **Cell 7 validates the cropper** against the refs of record and proves the head comes off a *mannequin* head before the arm is trusted.

**One session.** Needs on Drive under `v3_runs/`: `v34_ironman2_*.zip`, `v34_ironman2_bc_*.zip`, and optionally a previous `v35_linkC_*.zip` (which makes the `VEi`-based arms free).

In [ ]:
# 1 · settings
A100_USD_PER_HOUR = 0.689     # CAD/h at 5.3 CU/h x CAD 0.13/CU; edit if your rate differs
SEEDS = [46, 47, 48]          # the iron-man 2 seeds: every lock cell pairs with a verdict
ARMS = ("VEic", "M1qbc", "VEica", "M1qbca")
CROP_WORKERS = 4              # head crops run on threads: ORT/MediaPipe release the GIL
MATRIX = "v35_linkC.csv"      # 51 pairs: the union of the v3.4 lock's and v3.3's failures
DRIVE_PROJECT_DIR = "Side projects and shi"

In [ ]:
# 2 · Drive, the HF cache that holds klein, and the GPU opt-in for the V2 modules
import os
from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive'; BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
KLEIN = 'models--black-forest-labs--FLUX.2-klein-4B'
candidates = [os.path.join(MYDRIVE, 'hf_cache'), os.path.join(BASE, 'tryon_models', 'hf_cache'), os.path.join(BASE, 'hf_cache')]
found = [c for c in candidates if os.path.isdir(os.path.join(c, 'hub', KLEIN))]
os.environ['HF_HOME'] = found[0] if found else candidates[0]
os.environ['V3_MODEL_DIR'] = os.path.join(BASE if os.path.isdir(BASE) else MYDRIVE, 'v3_models')
os.makedirs(os.environ['HF_HOME'], exist_ok=True); os.makedirs(os.environ['V3_MODEL_DIR'], exist_ok=True)
# the V2 cropper's BiRefNet and human parser default to CPU - a 4-core-laptop decision from
# V2, and every V2 number on record was measured that way. Opt them onto this GPU; cell 6
# asserts it took and cell 7 validates the result against the refs of record.
os.environ['V2_ORT_GPU'] = '1'
print('HF_HOME      ', os.environ['HF_HOME'], '(klein cached)' if found else '(klein will download, ~9 GB)')
print('V3_MODEL_DIR ', os.environ['V3_MODEL_DIR'])
assert os.path.isdir(BASE), f'Drive project dir not found: {BASE}'

In [ ]:
# 3 · install, and get onnxruntime's CUDA provider ACTUALLY working
#
# The trap: pip installs the newest onnxruntime-gpu, which is built against CUDA 13, while
# this runtime's GPU stack is CUDA 12.x (torch decides that). The provider then registers -
# so get_available_providers() lists CUDAExecutionProvider and looks fine - and fails to
# dlopen at session creation, falling back to CPU with no error. The only honest test is to
# load the provider library itself, which is what the probe below does, in a subprocess so
# each candidate is tested in a clean interpreter.
!pip -q install -U diffusers transformers accelerate sentencepiece protobuf mediapipe opencv-contrib-python-headless
!pip -q uninstall -y onnxruntime onnxruntime-gpu >/dev/null 2>&1

probe = r"""
import ctypes, glob, os, site, sys
import onnxruntime as ort
dirs = sorted({d for p in site.getsitepackages() for d in glob.glob(os.path.join(p, 'nvidia', '*', 'lib'))})
os.environ['LD_LIBRARY_PATH'] = ':'.join(dirs + [os.environ.get('LD_LIBRARY_PATH', '')])
for d in dirs:                       # torch ships the CUDA 12 runtime; make it findable
    for f in os.listdir(d):
        if '.so' in f and any(k in f for k in ('cudart', 'cublas', 'cudnn', 'cufft', 'curand')):
            try: ctypes.CDLL(os.path.join(d, f), mode=ctypes.RTLD_GLOBAL)
            except OSError: pass
so = glob.glob(os.path.dirname(ort.__file__) + '/capi/libonnxruntime_providers_cuda.so')
if not so:
    print('NO_CUDA_PROVIDER_IN_WHEEL'); sys.exit(1)
ctypes.CDLL(so[0], mode=ctypes.RTLD_GLOBAL)      # raises OSError naming the missing lib
print('OK', ort.__version__)
"""
open('/content/ort_probe.py', 'w').write(probe)

import subprocess, sys, os
def probe_ok():
    r = subprocess.run([sys.executable, '/content/ort_probe.py'], capture_output=True, text=True)
    tail = (r.stdout + r.stderr).strip().splitlines()
    return r.returncode == 0, (tail[-1][:150] if tail else '')

CANDIDATES = ['', '==1.22.0', '==1.21.1', '==1.20.1', '==1.19.2', '==1.18.1']   # '' = newest
ORT_VERSION = None
for spec in CANDIDATES:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'onnxruntime-gpu{spec}'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  onnxruntime-gpu{spec or " (newest)"}: no installable wheel'); continue
    good, msg = probe_ok()
    print(f'  onnxruntime-gpu{spec or " (newest)"}: {"CUDA OK -> " + msg if good else "CPU only (" + msg + ")"}')
    if good:
        ORT_VERSION = msg.split()[-1]; break

# the parent kernel must see the same CUDA libs the probe preloaded
import ctypes, glob, site
_dirs = sorted({d for p in site.getsitepackages() for d in glob.glob(os.path.join(p, 'nvidia', '*', 'lib'))})
os.environ['LD_LIBRARY_PATH'] = ':'.join(_dirs + [os.environ.get('LD_LIBRARY_PATH', '')])
for _d in _dirs:
    for _f in os.listdir(_d):
        if '.so' in _f and any(k in _f for k in ('cudart', 'cublas', 'cudnn', 'cufft', 'curand')):
            try: ctypes.CDLL(os.path.join(_d, _f), mode=ctypes.RTLD_GLOBAL)
            except OSError: pass

!cd /content && rm -rf v35 && wget -q -O v35_bundle.zip https://github.com/101011101/magichour_takehome/raw/v3.3-lock/v35_linkC_bundle.zip && unzip -qo v35_bundle.zip -d v35
%cd /content/v35
import torch, onnxruntime as ort
for f in ('realesr-general-x4v3.pth', 'lib/run_v35_linkC.py', 'lib/ironman_bc_crop.py',
          'lib/phase3_variants.py', 'lib/garment_crop.py', 'v35_linkC.csv'):
    assert os.path.exists(f), f'bundle incomplete: {f}'
os.makedirs('run/inputs', exist_ok=True)
print(f'\n{torch.cuda.get_device_name(0)} | torch CUDA {torch.version.cuda} | onnxruntime {ort.__version__}')
assert torch.cuda.is_available(), 'no GPU - Runtime > Change runtime type > A100'
if ORT_VERSION:
    print(f'onnxruntime {ORT_VERSION} loads its CUDA provider - all five stages can be on the GPU')
else:
    print('NO onnxruntime build here loads CUDA. The crops will run on CPU (threaded).\n'
          'This is a Colab image mismatch, not a pipeline limit: in a controlled image you '
          'pin onnxruntime-gpu and CUDA to the same major and it works.')

In [ ]:
# 4 · everything already computed, off Drive. Nothing here is recomputed: these are the
#     records this arm is measured against, and redrawing them would break the pairing
#     with the human verdicts.
import glob, zipfile as zf, csv
rows = list(csv.DictReader(open(MATRIX)))
pairs = {r['set_id'] for r in rows}
stems = {r['person'] for r in rows} | {r['garment'] for r in rows}
garments = {r['garment'] for r in rows}
BASES = sorted({a.rstrip('ca') for a in ARMS})

WANT = ({f'inputs/{s}.jpg' for s in stems}
        | {f'inputs/{g}__A4.jpg' for g in garments}
        | {f'refs/{g}__{t}.jpg' for g in garments
           for t in ('VEi_small', 'VEi', 'BC', 'bald', 'VEi_headcut', 'M1qb_small',
                     'M1qb_headcut', 'VEic', 'VEica', 'M1qbc', 'M1qbca')}
        | {f'gen/{sid}__{a}__s{s}.jpg' for sid in pairs for s in SEEDS
           for a in ('VEi', 'BC', 'VEic', 'VEica', 'M1qbc', 'M1qbca')})

def pick(pat, exclude=None, required=True):
    zs = [z for z in sorted(glob.glob(os.path.join(BASE, 'v3_runs', pat)))
          if not (exclude and exclude in os.path.basename(z))]
    assert zs or not required, f'no {pat} on Drive under v3_runs/'
    return zs[-1] if zs else None

# '_bc_' sorts AFTER the digits, so a bare v34_ironman2_*.zip glob resolves to the BC zip
zips = [pick('v34_ironman2_*.zip', exclude='_bc_'), pick('v34_ironman2_bc_*.zip'),
        pick('v35_linkC_*.zip', required=False)]        # optional: a previous link C run
zips = [z for z in zips if z]
assert len(set(zips)) == len(zips), f'two patterns resolved to the same file: {zips}'

got = 0
for zp in zips:
    with zf.ZipFile(zp) as z:
        members = [n for n in z.namelist() if n in WANT]
        z.extractall('run', members=members); got += len(members)
    print(f'  {os.path.basename(zp)}: {len(members)} files')

n_small = len(glob.glob('run/refs/*__VEi_small.jpg')); n_bc = len(glob.glob('run/refs/*__BC.jpg'))
base_cells = len(glob.glob('run/gen/*__VEi__*.jpg')) + len(glob.glob('run/gen/*__BC__*.jpg'))
print(f'{got} files | VEi_small {n_small}/{len(garments)} · BC refs {n_bc}/{len(garments)} · '
      f'baseline cells {base_cells}/{2 * len(rows) * len(SEEDS)}')
todo_refs = sum(not os.path.exists(f'run/refs/{g}__{b}_small.jpg') for g in garments for b in BASES)
todo_edits = sum(not os.path.exists(f'run/gen/{r["set_id"]}__{a}__s{s}.jpg')
                 for r in rows for a in ARMS for s in SEEDS)
print(f'to compute: {todo_refs} references + {todo_edits} edits '
      f'= {todo_refs + todo_edits} klein calls (~{(todo_refs * 1.1 + todo_edits * 3.3) / 60:.0f} min)')
assert n_small == len(garments), "the lock's pre-SR references are incomplete - link C would redraw them"

In [ ]:
# 5 · load klein once, timed
import sys; sys.path.insert(0, 'lib')
import klein_local as K
K.load(); K.info()

In [ ]:
# 6 · GPU audit: every stage, on this hardware, measured rather than assumed.
#     Two of these were hard-coded to CPU in V2 and are the reason the crop stage used to
#     take ~80s an image on an A100 with the GPU idle.
import time, numpy as np, cv2, torch, onnxruntime as ort
import v3lib as L, run_ironman as R, garment_crop as GC, phase3_variants as P
paths = L.fetch_models(persist=os.environ.get('V3_MODEL_DIR'))
probe_g = sorted(garments)[0]
img = cv2.imread(f'run/inputs/{probe_g}.jpg')

t = time.time(); L.crop_a4(img, paths);            t_a4 = time.time() - t     # v3lib BiRefNet
t = time.time(); GC.biref_matte(img, 'audit', True); t_v2 = time.time() - t   # V2 BiRefNet
t = time.time(); P.parse_human(img);               t_ps = time.time() - t     # human parser
t = time.time(); R.to_1mp_sr(cv2.resize(img, (320, 480))); t_sr = time.time() - t

audit = [('klein (torch)',            K.info().get('gpu', '?'),                       None),
         ('SR upscaler (torch)',      R._SR.get('dev', '?'),                          t_sr),
         ('BiRefNet - A4 crop',       L._S.get('biref_prov', '?'),                    t_a4),
         ('BiRefNet - head crop (V2)', GC._STATE.get('biref_prov', '?'),              t_v2),
         ('human parser (V2)',        P._HP['m'].get_providers()[0] if P._HP.get('m') else 'unavailable', t_ps)]
print(f"{'stage':28s} {'device / provider':34s} seconds")
for name, dev, secs in audit:
    print(f'{name:28s} {str(dev):34s} {"" if secs is None else f"{secs:6.2f}"}')

on_gpu = lambda d: any(t in str(d) for t in ('CUDA', 'cuda', 'NVIDIA'))
bad = [n for n, d, _ in audit if not on_gpu(d)]
# klein and the SR pass MUST be on the GPU - everything else is a slow crop, not a wrong one
must = [n for n, d, _ in audit if not on_gpu(d) and ('klein' in n or 'SR' in n)]
assert not must, f'the generative stages are not on the GPU: {must}'
if bad:
    n_crops = len(garments) * len(BASES)
    print(f'\nWARNING - on CPU: {bad}')
    print(f'  onnxruntime 1.29 links its CUDA provider against CUDA 13; this runtime has '
          f'CUDA {torch.version.cuda}. The provider registers and then fails to load.')
    print(f'  {n_crops} head crops at ~{t_v2 + t_ps:.0f}s each = '
          f'~{n_crops * (t_v2 + t_ps) / 60:.0f} min sequential, '
          f'~{n_crops * (t_v2 + t_ps) / 60 / CROP_WORKERS:.0f} min on {CROP_WORKERS} threads.')
else:
    print('\nall five stages on the GPU')

In [ ]:
# 7 · the cropper, validated - then the case that has never been proved: does the head
#     come off a generated MANNEQUIN head? BC only ever cropped a bald photograph.
import glob
import ironman_bc_crop as C
from IPython.display import display
from PIL import Image

def side_by_side(imgs, h=440):
    fit = [cv2.resize(i, (max(1, int(i.shape[1] * h / i.shape[0])), h)) for i in imgs if i is not None]
    w = max(f.shape[1] for f in fit)
    return Image.fromarray(cv2.cvtColor(np.hstack(
        [np.pad(f, ((0, 0), (0, w - f.shape[1]), (0, 0)), constant_values=255) for f in fit]),
        cv2.COLOR_BGR2RGB))

bad = 0
for vp in sorted(glob.glob('validation/*__BC.jpg')):
    stem = os.path.basename(vp)[:-len('__BC.jpg')]
    src = f'run/refs/{stem}__bald.jpg'
    if not os.path.exists(src):
        continue
    a = cv2.imread(vp); b, _ = C.crop_bc(cv2.imread(src), f'val_{stem}')
    if abs(a.shape[0] - b.shape[0]) > 8 or abs(a.shape[1] - b.shape[1]) > 8:
        print(f'  validate {stem}: SHAPE {a.shape[:2]} vs {b.shape[:2]}'); bad += 1; continue
    mad = float(np.abs(a.astype(np.float32) - cv2.resize(b, (a.shape[1], a.shape[0])).astype(np.float32)).mean())
    print(f'  validate {stem}: MAD {mad:.2f}')
    bad += mad > 4.0
assert not bad, 'the cropper does not reproduce the refs of record on this machine'

small = cv2.imread(f'run/refs/{probe_g}__VEi_small.jpg')
t0 = time.time(); cut, cranium = C.crop_bc(small, f'probe_{probe_g}'); secs = time.time() - t0
print(f'\n{probe_g}: {small.shape[1]}x{small.shape[0]} -> {cut.shape[1]}x{cut.shape[0]}  '
      f'cranium_used={cranium}  {secs:.1f}s  (CPU-only took ~80s an image)')
display(side_by_side([cv2.imread(f'run/inputs/{probe_g}__A4.jpg'), small, cut]))
assert cranium, 'the parser did not fire on the mannequin head - look before running the arm'

In [ ]:
# 8 · one pair end to end, before paying for the set
import run_v35_linkC as V
V.main(MATRIX, 'testset', seeds=SEEDS[:1], arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR, crop_workers=CROP_WORKERS, limit=1)
print(sorted(f for f in os.listdir('run/gen') if any(f'__{a}__' in f for a in ARMS))[:8])

In [ ]:
# 9 · the run (resumable - rerun after any disconnect and it skips what is on disk)
import run_v35_linkC as V, json
V.main(MATRIX, 'testset', seeds=SEEDS, arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR, crop_workers=CROP_WORKERS)
print(json.dumps(json.load(open('run/meta/cost_v35.json')), indent=1))

In [ ]:
# 10 · every cell landed? the parser fired? where was the ankle cut a no-op?
import json, glob
meta = json.load(open('run/meta/prompts_v35.json'))
want = len(rows) * len(SEEDS)
missed = sorted({g for g in garments for b in BASES
                 if not meta.get(g, {}).get(f'{b}_cranium_used')})
print('references where the parser did NOT fire:', missed or 'none')
cut_refs = [(g, a) for g in garments for a in ARMS if a.endswith('a')]
noop = [(g, a) for g, a in cut_refs if meta.get(g, {}).get(f'{a}_ankle_row') is None]
print(f'ankle cut a no-op (no ankles in frame) on {len(noop)} of {len(cut_refs)} cut references')
short = []
for a in list(ARMS) + ['VEi', 'BC']:
    n = len(glob.glob(f'run/gen/*__{a}__*.jpg'))
    print(f'  {a:7s} {n}/{want} cells')
    if n < want: short.append(a)
assert not short, f'incomplete arms: {short} - rerun cell 9, it resumes'
print('\ncomplete:', (len(ARMS) + 2) * want, 'cells across', len(ARMS) + 2, 'arms')
if missed:
    print('NOTE:', len(missed), 'garment(s) fell back off the parser; their head crops are '
          'flagged in the meta and read separately on the page, not averaged in.')

In [ ]:
# 11 · zip references, outputs and meta to Drive
import shutil, time, zipfile
name = f"v35_linkC_{time.strftime('%Y%m%d_%H%M')}"
KEEP = tuple(f'__{a}.' for a in ARMS) + ('__VEi_small', '__M1qb_small', '__VEi_headcut',
                                         '__M1qb_headcut', '__BC.', '__VEi.')
src = 'run'
with zipfile.ZipFile(f'/content/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    kept = [f for f in os.listdir(f'{src}/refs') if any(t in f for t in KEEP)]
    for f in kept:                       z.write(f'{src}/refs/{f}',   'refs/' + f)
    for f in os.listdir(f'{src}/gen'):   z.write(f'{src}/gen/{f}',    'gen/' + f)
    for f in os.listdir(f'{src}/inputs'): z.write(f'{src}/inputs/{f}', 'inputs/' + f)
    for f in os.listdir(f'{src}/meta'):  z.write(f'{src}/meta/{f}',   'meta/' + f)
# verify the archive before trusting it: it is the only thing that leaves this session
with zipfile.ZipFile(f'/content/{name}.zip') as z:
    assert z.testzip() is None, 'the zip is corrupt'
    names = z.namelist()
    n_gen = sum(n.startswith('gen/') for n in names)
    for need in ('meta/prompts_v35.json', 'meta/run_v35.json', 'meta/cost_v35.json'):
        assert need in names, f'missing {need} from the zip'
    assert n_gen >= (len(ARMS) + 2) * want, f'only {n_gen} generated cells in the zip'
os.makedirs(os.path.join(BASE, 'v3_runs'), exist_ok=True)
shutil.copy(f'/content/{name}.zip', os.path.join(BASE, 'v3_runs', f'{name}.zip'))
print(f'{name}.zip · {len(names)} files ({n_gen} cells, {len(kept)} refs) · '
      f'{os.path.getsize(f"/content/{name}.zip") / 1e6:.0f} MB -> Drive v3_runs/')
try:
    from google.colab import files; files.download(f'/content/{name}.zip')
except Exception as e:
    print('download it from Drive:', e)